In [2]:
import pandas as pd
import sqlite3

movies = pd.read_csv("movies.csv")
genres = pd.read_csv("genres.csv")
movie_genres = pd.read_csv("movie_genres.csv")
movie_keywords = pd.read_csv("movie_keywords.csv")
cast = pd.read_csv("cast.csv")
crew = pd.read_csv("crew.csv")

conn = sqlite3.connect("tmdb_movies.db")

movies.to_sql("movies", conn, if_exists="replace", index=False)
genres.to_sql("genres", conn, if_exists="replace", index=False)
movie_genres.to_sql("movie_genres", conn, if_exists="replace", index=False)
movie_keywords.to_sql("movie_keywords", conn, if_exists="replace", index=False)
cast.to_sql("cast", conn, if_exists="replace", index=False)
crew.to_sql("crew", conn, if_exists="replace", index=False)


7289

In [4]:
print("MOVIES COLUMNS:")
print(movies.columns.tolist())

print("\nGENRES COLUMNS:")
print(genres.columns.tolist())

print("\nMOVIE_GENRES COLUMNS:")
print(movie_genres.columns.tolist())

MOVIES COLUMNS:
['movie_id', 'title', 'original_language', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'status', 'overview']

GENRES COLUMNS:
['genre_id', 'genre_name']

MOVIE_GENRES COLUMNS:
['movie_id', 'genre_id']


In [3]:
query = """
SELECT DISTINCT
    m.title,
    m.release_date
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
AND m.release_date >= '2016-01-01'
ORDER BY m.release_date;
"""

df = pd.read_sql_query(query, conn)

df

,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Zoolander 2,2016-02-06
3,Deadpool,2016-02-09
4,Triple 9,2016-02-19
...,...,...
263,F1,2025-06-25
264,Superman,2025-07-09
265,The Fantastic 4: First Steps,2025-07-23
266,Predator: Badlands,2025-11-05


In [4]:
query = """
SELECT
    m.title,
    m.revenue,
    GROUP_CONCAT(g.genre_name, ', ') AS genres
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df

,title,revenue,genres
0,Avatar,2.923706e+09,"Adventure, Action, Science Fiction"
1,Avengers: Endgame,2.799439e+09,"Adventure, Action, Science Fiction"
2,Avatar: The Way of Water,2.334485e+09,"Adventure, Action, Science Fiction"
3,Titanic,2.264162e+09,"Drama, Romance"
4,Star Wars: The Force Awakens,2.068224e+09,"Adventure, Action, Science Fiction"
5,Avengers: Infinity War,2.052415e+09,"Adventure, Action, Science Fiction"
6,Spider-Man: No Way Home,1.921207e+09,"Adventure, Action, Science Fiction"
7,Zootopia 2,1.868209e+09,"Adventure, Animation, Comedy, Mystery, Family"
8,Inside Out 2,1.698864e+09,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1.671537e+09,"Adventure, Thriller, Science Fiction"


In [ ]:
query = """
SELECT
    g.genre_name,
    ROUND(AVG(m.budget), 2) AS average_budget,
    ROUND(AVG(m.revenue), 2) AS average_revenue
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY average_revenue DESC;
"""

df = pd.read_sql_query(query, conn)

df

,genre_name,average_budget,average_revenue
0,Adventure,1.093600e+08,3.902145e+08
1,Animation,8.336061e+07,3.597435e+08
2,Family,8.458426e+07,3.430968e+08
3,Science Fiction,9.525992e+07,3.235601e+08
4,Action,9.329457e+07,3.000547e+08
5,Fantasy,8.617784e+07,2.942975e+08
6,Comedy,5.256907e+07,2.113513e+08
7,Music,4.106939e+07,1.974526e+08
8,Romance,3.461113e+07,1.627896e+08
9,Thriller,4.938770e+07,1.605152e+08


In [6]:
query = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM cast
GROUP BY actor_name
ORDER BY movie_count DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Willem Dafoe,32
6,Scarlett Johansson,32
7,Mark Wahlberg,32
8,Tom Cruise,31
9,Morgan Freeman,31


In [7]:
query = """
SELECT
    person_name AS director_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE LOWER(job) = 'director'
GROUP BY person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC;
"""

df = pd.read_sql_query(query, conn)

df

,director_name,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Andrew Adamson,4
213,Alfonso Cuarón,4
214,Alexandre Aja,4
215,Adam Wingard,4


In [8]:
query = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
GROUP BY keyword_name
ORDER BY movie_count DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


In [25]:
query = """
SELECT
    title,
    budget,
    revenue
FROM movies
WHERE budget > (
    SELECT AVG(budget)
    FROM movies
    WHERE budget > 0
)
AND revenue < (
    SELECT AVG(revenue)
    FROM movies
    WHERE revenue > 0
)
ORDER BY budget DESC;
"""

df = pd.read_sql_query(query, conn)

df

,title,budget,revenue
0,The Marvels,274800000.0,206136825.0
1,Red One,250000000.0,185700759.0
2,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,Wonder Woman 1984,200000000.0,169601036.0
4,Luca,200000000.0,51074773.0
...,...,...,...
283,Rambo III,63000000.0,189015611.0
284,The Equalizer 2,62000000.0,190400157.0
285,Æon Flux,62000000.0,53321673.0
286,Shooter,61000000.0,95700000.0


In [9]:
query = """
SELECT DISTINCT
    c.actor_name,
    cr.person_name AS director_name,
    m.title AS movie_title
FROM cast AS c
JOIN crew AS cr
    ON c.movie_id = cr.movie_id
JOIN movies AS m
    ON c.movie_id = m.movie_id
WHERE LOWER(cr.job) = 'director'
AND LOWER(cr.person_name) = 'christopher nolan'
ORDER BY m.title, c.actor_name;
"""

df = pd.read_sql_query(query, conn)

df

,actor_name,director_name,movie_title
0,Christian Bale,Christopher Nolan,Batman Begins
1,Cillian Murphy,Christopher Nolan,Batman Begins
2,Gary Oldman,Christopher Nolan,Batman Begins
3,Katie Holmes,Christopher Nolan,Batman Begins
4,Ken Watanabe,Christopher Nolan,Batman Begins
...,...,...,...
105,Michael Caine,Christopher Nolan,The Prestige
106,Piper Perabo,Christopher Nolan,The Prestige
107,Rebecca Hall,Christopher Nolan,The Prestige
108,Samantha Mahurin,Christopher Nolan,The Prestige


In [10]:
query = """
SELECT
    g.genre_name,
    ROUND(AVG(m.vote_average), 2) AS average_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY average_rating DESC
LIMIT 1;
"""

df = pd.read_sql_query(query, conn)

df

,genre_name,average_rating,movie_count
0,War,7.36,85


In [11]:
query = """
SELECT m.title,
       COUNT(mk.keyword_name) AS keyword_count
FROM movies AS m
JOIN movie_keywords AS mk
ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
df

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,Pacific Rim: Uprising,49
